# Add: Transformer with Attention Layer

In [1]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [2]:
VOCAB_SIZE = 1000
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [3]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [4]:
_ = torch.manual_seed(123)

In [22]:
def separate_dot(text):
    text = re.sub(r'([.,!?])', r' \1 ', text)
    return text.strip()

text_data = [separate_dot(text) for text in data]

In [18]:
example = text_data[0]
print(example)

She composes songs and practices piano daily .


In [24]:
vocab = set()

for text in text_data:
    for word in text.split():
       vocab.add(word)

vocab = sorted(list(vocab))
print(len(vocab))


70


In [25]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: navigation
38: nearby
39: newspaper
40: novel
41: novels
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects
48: puzzles
49: reads
50: regularly
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars
58: studies
59: teaches
60: the
61: trains
62: trips
63: tunes
64: using
65: weekend
66: wildlife
67: with
68: wooden
69: writes


In [26]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

In [36]:
def text_to_token_ids(text, word_to_idx):
    token = text.split()

    return [word_to_idx[t] for t in token]

def token_to_text(token_ids, idx_to_word):
    return " ".join([idx_to_word[idx] for idx in token_ids])

In [37]:
print(example)

She composes songs and practices piano daily .


In [43]:
text_to_token_ids(example,word_to_idx)


[2, 11, 55, 3, 46, 45, 14, 0]

In [44]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN + 1]

[2, 11, 55, 3, 46, 45, 14]

In [ ]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN]

[2, 11, 55, 3, 46, 45]

In [46]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN - 1]

[2, 11, 55, 3, 46]

In [39]:
token_to_text(text_to_token_ids(example,word_to_idx), idx_to_word)

'She composes songs and practices piano daily .'

In [54]:
class LLMDataset(Dataset):
    def __init__(self, text_data, word_to_idx, context_len):
        self.text_data = text_data
        self.word_to_idx = word_to_idx
        self.context_len = context_len


    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.text_data[idx], self.word_to_idx))[: self.context_len + 1]
        x = tokens[:-1]
        y = tokens[1:]
        return x,y
    
triain_dataset = LLMDataset(text_data, word_to_idx, CONTEXT_LEN)
train_dataloader = DataLoader(triain_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_dataloader))

4


In [58]:
example_input_output = next(iter(train_dataloader))
print(example_input_output)
print ("-------------------------")
example_input_output[1][0]

[tensor([[ 2, 11, 55,  3, 46, 45],
        [ 1, 49,  5,  3, 22, 60],
        [ 1, 49, 40,  3,  8, 35],
        [ 2, 11, 55,  3, 69, 41]]), tensor([[11, 55,  3, 46, 45, 14],
        [49,  5,  3, 22, 60, 38],
        [49, 40,  3,  8, 35, 18],
        [11, 55,  3, 69, 41,  0]])]
-------------------------


tensor([11, 55,  3, 46, 45, 14])

In [59]:
print ("-------------------------")
example_input_output[0][0]


-------------------------


tensor([ 2, 11, 55,  3, 46, 45])

In [60]:
print ("-------------------------")
example_input_output[0][1]

-------------------------


tensor([ 1, 49,  5,  3, 22, 60])

In [61]:
class Attantion(nn.Module):
    def __init__(self,d_in,d_out,context_length):
        super().__init__()
        self.d_out = d_out
        self.W_q = nn.Linear(d_in, d_out,bias=False)
        self.W_k = nn.Linear(d_in, d_out,bias=False)
        self.W_v = nn.Linear(d_in, d_out,bias=False)

    def forward(self, x, return_weights=False):
        _,num_tokens,_ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attn_scores = Q @ K.transpose(1, 2) 
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores = attn_scores.masked_fill(~mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores/(self.d_out **0.5), dim=-1)
        context_vec = attn_weights @ V

        if return_weights:
            return context_vec, attn_weights

        return context_vec